In [ ]:
# Run from the repository root so data/, scripts/, dashboard/, outputs/ paths resolve.
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Outcome-Structure Analysis: PCA and Within/Between-Operator Coupling

**Purpose.** Reproduce the outcome-structure statistics reported in the Results (the
"shared error-and-uncertainty axis" paragraph): the principal-components analysis of the
three trial-level outcomes (accuracy, response latency, confidence) and the
within- versus between-operator decomposition of their couplings.

**Scope.** Sign and Animal visual-identification probes only (the modelling scope, *n* = 439
probes across 37 operators).

**Data.** `master_routes_1_to_6_latency.csv`, columns `Target_Accuracy` (1 = correct),
`Target_Latency` (s), `Target_Confidence` (1-7), `Participant_ID`, `Question_Type`.

Outputs feed directly into the manuscript sentence and the supplementary note.

**Packages & method.** scikit-learn `PCA` on the standardised outcomes and scipy `spearmanr` for the rank correlations; pandas / numpy for the within- versus between-operator decomposition (Spearman on operator-demeaned residuals versus operator means). PCA is used descriptively, to show the three outcomes load on one shared axis.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option('display.width', 120)
OUTCOMES = ['Target_Accuracy', 'Target_Latency', 'Target_Confidence']

## 1. Load and filter to the Sign and Animal probes

In [2]:
df = pd.read_csv('data/master_routes_1_to_6_latency.csv')

d = df[df['Question_Type'].isin(['Sign', 'Animal'])].copy()
for c in OUTCOMES:
    d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=OUTCOMES + ['Participant_ID'])

print(f"Probes analysed : {len(d)}")
print(f"Operators       : {d['Participant_ID'].nunique()}")
print(f"Probe mix       : {d['Question_Type'].value_counts().to_dict()}")

Probes analysed : 439
Operators       : 37
Probe mix       : {'Sign': 221, 'Animal': 218}


## 2. Raw trial-level Spearman correlations

Rank correlations among the three outcomes, pooled across all probes. These are the
headline associations quoted in the paragraph (slower responses are less confident and
less accurate).

In [3]:
acc, lat, conf = d['Target_Accuracy'], d['Target_Latency'], d['Target_Confidence']

raw = {
    'latency ~ confidence': spearmanr(lat, conf).correlation,
    'latency ~ accuracy'  : spearmanr(lat, acc).correlation,
    'accuracy ~ confidence': spearmanr(acc, conf).correlation,
}
for k, v in raw.items():
    print(f"{k:22s}: rho = {v:+.3f}")

latency ~ confidence  : rho = -0.498
latency ~ accuracy    : rho = -0.313
accuracy ~ confidence : rho = +0.476


## 3. Principal-components analysis of the three outcomes

PCA on the standardised (z-scored) outcomes, i.e. on the correlation matrix. The first
component is the "shared error-and-uncertainty axis"; its variance-explained is the
`~61%` figure, and its loadings show the axis runs from correct / confident / fast to
wrong / unsure / slow.

In [4]:
X = StandardScaler().fit_transform(d[OUTCOMES])
pca = PCA().fit(X)

evr = pca.explained_variance_ratio_
print("Explained variance ratio :", np.round(evr, 3))
print(f"PC1 variance explained   : {100*evr[0]:.1f}%")

load = pd.Series(pca.components_[0], index=['accuracy', 'latency', 'confidence'])
print("\nPC1 loadings:")
print(load.round(3).to_string())

Explained variance ratio : [0.613 0.248 0.139]
PC1 variance explained   : 61.3%

PC1 loadings:
accuracy      0.586
latency      -0.503
confidence    0.636


## 4. Within- versus between-operator decomposition

Each coupling is split into a **between-operator** part (correlation of operator means,
37 points) and a **within-operator** part (correlation of operator-demeaned residuals,
i.e. trial-level deviations from each operator's own mean). Spearman is used to match the
raw correlations above.

This separates *who the operator is* from *what happens on a given trial*: the
accuracy-latency link is carried mainly between operators, whereas the latency-confidence
link is a within-operator, trial-level phenomenon.

In [5]:
def within_between(sub, a, b):
    g = sub.groupby('Participant_ID')
    means = g[[a, b]].mean()
    between = spearmanr(means[a], means[b]).correlation
    resid = sub[[a, b]] - g[[a, b]].transform('mean')
    within = spearmanr(resid[a], resid[b]).correlation
    return within, between

rows = []
for a, b, label in [('Target_Latency', 'Target_Accuracy', 'latency ~ accuracy'),
                    ('Target_Latency', 'Target_Confidence', 'latency ~ confidence')]:
    w, bt = within_between(d, a, b)
    rows.append({'coupling': label, 'within_operator': round(w, 3),
                 'between_operator': round(bt, 3),
                 'carried_by': 'within (trial-level)' if abs(w) > abs(bt) else 'between (operator)'})

decomp = pd.DataFrame(rows)
print(decomp.to_string(index=False))

            coupling  within_operator  between_operator           carried_by
  latency ~ accuracy           -0.227            -0.484   between (operator)
latency ~ confidence           -0.501            -0.316 within (trial-level)


## 5. Mapping to the manuscript

| Statistic in prose | Value here |
|---|---|
| PC1 = ~61% of joint variance | see PC1 above |
| latency-confidence Spearman -0.50 | raw table |
| latency-accuracy Spearman -0.31 | raw table |
| accuracy-latency between-operator -0.48 | decomposition table |
| accuracy-latency within-operator -0.23 | decomposition table |

The PC1 loadings (accuracy +, latency -, confidence +) define the shared axis; latency's
lower absolute loading and its within-operator coupling with confidence are what make it
"partly separable" and justify retaining a distinct latency detector.

In [6]:
# One-line reproducibility summary
summary = {
    'n_probes': int(len(d)),
    'n_operators': int(d['Participant_ID'].nunique()),
    'pc1_variance_explained_pct': round(100 * evr[0], 1),
    'spearman_latency_confidence': round(raw['latency ~ confidence'], 3),
    'spearman_latency_accuracy': round(raw['latency ~ accuracy'], 3),
}
summary

{'n_probes': 439,
 'n_operators': 37,
 'pc1_variance_explained_pct': 61.3,
 'spearman_latency_confidence': -0.498,
 'spearman_latency_accuracy': -0.313}